In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp



In [ ]:
# ==============================================================================
# PHASE 2: Enterprise Data Pipeline (Mock Central Catalog -> Databricks -> Unity Catalog)


In [ ]:
# ------------------------------------------------------------------------------
print("="*70)
print("SCOPE NOTE (read before demoing to stakeholders)")
print("="*70)
print(
    "This POC runs entirely on open-source, locally-emulated components:\n"
    "OSS Unity Catalog, an Iceberg REST catalog standing in for AWS Glue,\n"
    "and LocalStack for S3. Some failures encountered during development\n"
    "(e.g. Delta UniForm's IcebergConverter against an embedded metastore)\n"
    "are specific to this OSS/local tooling combination and have NOT yet\n"
    "been confirmed on managed Databricks Unity Catalog + real AWS Glue.\n"
    "See README.md's 'Validation Still Needed' section before treating the\n"
    "conclusions here as final for the production architecture decision."
)
print("="*70 + "\n")


In [ ]:
# ==============================================================================
print("Initializing Phase 2 Pipeline...")

# Initialize a unified Spark session with both Iceberg and Delta Lake capabilities.
# This simulates the rich Databricks runtime environment.
#
# VERSION PIN NOTE: delta-spark/delta-iceberg are pinned to 3.2.0, not 3.2.1.
# This base image (jupyter/pyspark-notebook:spark-3.5.0) ships Spark 3.5.0.
# delta-spark 3.2.1's compiled bytecode references a Catalyst-internal
# ExpressionSet method signature that changed between Spark 3.5.0/3.5.1 and
# 3.5.3, causing java.lang.NoSuchMethodError at read time (confirmed exact
# match: delta-io/delta#3737 -- "Spark 3.5.1 and delta 3.2.0 work fine";
# "Spark 3.5.3 and delta 3.2.1 works fine"; 3.5.0/3.5.1 + 3.2.1 does not).
# We no longer need anything 3.2.1-specific since the UniForm/IcebergCompatV2
# path was abandoned in favor of the add_files bridge, so 3.2.0 is the
# correct, lower-risk pin for this Spark version rather than rebuilding the
# image for Spark 3.5.3.
spark = SparkSession.builder \
    .appName("Databricks-POC-Simulator") \
    .config("spark.jars.packages", 
            "io.unitycatalog:unitycatalog-spark_2.12:0.2.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
            "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
            "io.delta:delta-spark_2.12:3.2.0,"
            "io.delta:delta-iceberg_2.12:3.2.0," \
            "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", 
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.catalog.iceberg_cat", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg_cat.type", "hive") \
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity.warehouse", "s3a://lakehouse-bucket/unity_catalog/") \
    .config("spark.sql.catalog.unity.uri", "http://unity-catalog-server:8080") \
    .config("spark.sql.catalog.central_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.central_catalog.catalog-impl", "org.apache.iceberg.rest.RESTCatalog") \
    .config("spark.sql.catalog.central_catalog.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.central_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.central_catalog.warehouse", "s3a://lakehouse-bucket/central_warehouse/") \
    .config("spark.hadoop.hive.metastore.schema.verification", "false") \
    .config("spark.hadoop.hive.support.concurrency", "false") \
    .config("spark.hadoop.hive.txn.manager", "org.apache.hadoop.hive.ql.lockmgr.DummyTxnManager") \
    .config("spark.hadoop.datanucleus.schema.autoCreateAll", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.endpoint", "http://localstack:4566") \
    .config("spark.sql.catalog.central_catalog.s3.path-style-access", "true") \
    .config("spark.sql.catalog.central_catalog.client.region", "us-east-1") \
    .config("spark.sql.catalog.central_catalog.s3.access-key-id", "test") \
    .config("spark.sql.catalog.central_catalog.s3.secret-access-key", "test") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2A: Originate Central Data (Iceberg REST Mock)


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2A] Originating Source Data in Central Catalog ---")
spark.sql("CREATE NAMESPACE IF NOT EXISTS central_catalog.default")

data = [("1", "Alice", 1000), ("2", "Bob", 2000), ("3", "Charlie", 3000)]
columns = ["account_id", "name", "balance"]
df_source = spark.createDataFrame(data, columns)

df_source.writeTo("central_catalog.default.source_accounts").using("iceberg").createOrReplace()
print("Successfully created 'source_accounts' in Central Catalog (Iceberg).")



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2B & 2C: Federated Read & Data Manipulation


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2B & 2C] Federated Read & Data Manipulation (Databricks) ---")
# Simulating Databricks reading the Central table via federation
df_federated = spark.table("central_catalog.default.source_accounts")

df_transformed = df_federated \
    .withColumn("balance_with_interest", col("balance") * 1.05) \
    .withColumn("processed_at", current_timestamp())

print("Data transformation complete. Schema:")
df_transformed.printSchema()



In [ ]:
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("SIMULATION BOUNDARY NOTE -- Phase 2B/2C (Federated Read)")
print("="*70)
print(
    "This step queries 'central_catalog' directly via Spark's own Iceberg\n"
    "REST catalog plugin. It does NOT go through Unity Catalog's Lakehouse\n"
    "Federation feature -- that governance layer (foreign/federated catalog\n"
    "connections, credential vending, permissions, lineage) is a managed-\n"
    "Databricks capability not present in Unity Catalog OSS. In production,\n"
    "this read would instead be mediated by UC's federation layer. Flagging\n"
    "this explicitly so it isn't mistaken for a validated federation test."
)
print("="*70 + "\n")


In [ ]:
# ------------------------------------------------------------------------------
# Phase 2D: Write to Unity Catalog (Delta UniForm)


In [ ]:
# WORKAROUND: Create missing Hive transaction tables in Derby so Iceberg's stubborn HiveCatalog succeeds
def create_derby_tables(spark):
    try:
        sc = spark.sparkContext
        DriverManager = sc._gateway.jvm.java.sql.DriverManager
        conn = DriverManager.getConnection("jdbc:derby:/home/jovyan/work/metastore_db;create=true")
        stmt = conn.createStatement()
        queries = [
            "CREATE TABLE NEXT_LOCK_ID (NL_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_LOCK_ID VALUES (1)",
            "CREATE TABLE HIVE_LOCKS (HL_LOCK_EXT_ID BIGINT NOT NULL, HL_LOCK_INT_ID BIGINT NOT NULL, HL_TXNID BIGINT NOT NULL, HL_DB VARCHAR(128) NOT NULL, HL_TABLE VARCHAR(128), HL_PARTITION VARCHAR(767), HL_LOCK_STATE CHAR(1) NOT NULL, HL_LOCK_TYPE CHAR(1) NOT NULL, HL_LAST_HEARTBEAT BIGINT NOT NULL, HL_ACQUIRED_AT BIGINT, HL_USER VARCHAR(128) NOT NULL, HL_HOST VARCHAR(128) NOT NULL, HL_HEARTBEAT_COUNT INT, HL_AGENT_INFO VARCHAR(128), HL_BLOCKEDBY_EXT_ID BIGINT, HL_BLOCKEDBY_INT_ID BIGINT, PRIMARY KEY(HL_LOCK_EXT_ID, HL_LOCK_INT_ID))",
            "CREATE TABLE TXNS (TXN_ID BIGINT NOT NULL, TXN_STATE CHAR(1) NOT NULL, TXN_STARTED BIGINT NOT NULL, TXN_LAST_HEARTBEAT BIGINT NOT NULL, TXN_USER VARCHAR(128) NOT NULL, TXN_HOST VARCHAR(128) NOT NULL, TXN_AGENT_INFO VARCHAR(128), TXN_META_INFO VARCHAR(128), TXN_HEARTBEAT_COUNT INT, TXN_TYPE INT, PRIMARY KEY(TXN_ID))",
            "CREATE TABLE NEXT_TXN_ID (NTXN_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_TXN_ID VALUES (1)",
            "CREATE TABLE AUX_TABLE (MT_KEY1 VARCHAR(128) NOT NULL, MT_KEY2 BIGINT NOT NULL, MT_COMMENT VARCHAR(255), PRIMARY KEY(MT_KEY1, MT_KEY2))",
            "CREATE TABLE WRITE_SET (WS_DATABASE VARCHAR(128) NOT NULL, WS_TABLE VARCHAR(128) NOT NULL, WS_PARTITION VARCHAR(767), WS_TXNID BIGINT NOT NULL, WS_COMMIT_ID BIGINT NOT NULL, WS_OPERATION_TYPE CHAR(1) NOT NULL)",
            "CREATE TABLE TXN_COMPONENTS (TC_TXNID BIGINT NOT NULL, TC_DATABASE VARCHAR(128) NOT NULL, TC_TABLE VARCHAR(128), TC_PARTITION VARCHAR(767), TC_OPERATION_TYPE CHAR(1) NOT NULL, TC_WRITEID BIGINT)",
            "CREATE TABLE COMPLETED_TXN_COMPONENTS (CTC_TXNID BIGINT NOT NULL, CTC_DATABASE VARCHAR(128) NOT NULL, CTC_TABLE VARCHAR(256), CTC_PARTITION VARCHAR(767), CTC_TIMESTAMP timestamp DEFAULT CURRENT_TIMESTAMP NOT NULL, CTC_WRITEID BIGINT, CTC_UPDATE_DELETE CHAR(1) NOT NULL)",
            "CREATE TABLE COMPACTION_QUEUE (CQ_ID BIGINT NOT NULL, CQ_DATABASE VARCHAR(128) NOT NULL, CQ_TABLE VARCHAR(128) NOT NULL, CQ_PARTITION VARCHAR(767), CQ_STATE CHAR(1) NOT NULL, CQ_TYPE CHAR(1) NOT NULL, CQ_TBLPROPERTIES VARCHAR(2048), CQ_WORKER_ID VARCHAR(128), CQ_START BIGINT, CQ_RUN_AS VARCHAR(128), CQ_HIGHEST_WRITE_ID BIGINT, CQ_META_INFO VARCHAR(2048) FOR BIT DATA, CQ_HADOOP_JOB_ID VARCHAR(32), PRIMARY KEY(CQ_ID))",
            "CREATE TABLE COMPLETED_COMPACTIONS (CC_ID BIGINT NOT NULL, CC_DATABASE VARCHAR(128) NOT NULL, CC_TABLE VARCHAR(128) NOT NULL, CC_PARTITION VARCHAR(767), CC_STATE CHAR(1) NOT NULL, CC_TYPE CHAR(1) NOT NULL, CC_TBLPROPERTIES VARCHAR(2048), CC_WORKER_ID VARCHAR(128), CC_START BIGINT, CC_END BIGINT, CC_RUN_AS VARCHAR(128), CC_HIGHEST_WRITE_ID BIGINT, CC_META_INFO VARCHAR(2048) FOR BIT DATA, CC_HADOOP_JOB_ID VARCHAR(32), PRIMARY KEY(CC_ID))",
            "CREATE TABLE NEXT_COMPACTION_QUEUE_ID (NCQ_NEXT BIGINT NOT NULL)",
            "INSERT INTO NEXT_COMPACTION_QUEUE_ID VALUES (1)",
            "CREATE TABLE TXN_TO_WRITE_ID (T2W_TXNID BIGINT NOT NULL, T2W_DATABASE VARCHAR(128) NOT NULL, T2W_TABLE VARCHAR(256) NOT NULL, T2W_WRITEID BIGINT NOT NULL)",
            "CREATE TABLE NEXT_WRITE_ID (NWI_DATABASE VARCHAR(128) NOT NULL, NWI_TABLE VARCHAR(256) NOT NULL, NWI_NEXT BIGINT NOT NULL)",
            "CREATE TABLE MIN_HISTORY_LEVEL (MHL_TXNID BIGINT NOT NULL, MHL_MIN_OPEN_TXNID BIGINT NOT NULL, PRIMARY KEY(MHL_TXNID))",
            "CREATE TABLE MATERIALIZATION_REBUILD_LOCKS (MRL_TXN_ID BIGINT NOT NULL, MRL_DB_NAME VARCHAR(128) NOT NULL, MRL_TBL_NAME VARCHAR(256) NOT NULL, MRL_LAST_HEARTBEAT BIGINT NOT NULL, PRIMARY KEY(MRL_TXN_ID))",
            "CREATE TABLE IUD_CACHE (IC_DB VARCHAR(128) NOT NULL, IC_TABLE VARCHAR(128) NOT NULL, IC_PARTITION VARCHAR(767), IC_DELETED_RECORDS VARCHAR(4000), IC_UPDATED_RECORDS VARCHAR(4000))"
        ]
        for q in queries:
            try:
                stmt.executeUpdate(q)
            except:
                pass
        try:
            stmt.executeUpdate("DELETE FROM HIVE_LOCKS")
            stmt.executeUpdate("DELETE FROM TXNS")
        except:
            pass
        print("Created missing Derby tables for Hive lock manager.")
    except Exception as e:
        print("Failed to connect to Derby:", e)

create_derby_tables(spark)


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2D] Writing to Unity Catalog ---")
uc_table_path = "s3://lakehouse-bucket/unity_catalog/transformed_accounts"

# 1. Write plain Delta to S3 (no UniForm, no column mapping UUIDs!)
df_transformed.write.format("delta").mode("overwrite").save(uc_table_path)
print("Successfully wrote raw Delta files to S3.")

# 2. Register directly in Unity Catalog via REST API
print("\nRegistering table in Unity Catalog via REST API...")
import requests
schema = df_transformed.schema

def map_type_name(t):
    name = t.typeName().upper()
    if name == 'INTEGER':
        return 'INT'
    return name

columns = [
    {
        "name": f.name,
        "type_text": f.dataType.simpleString(),
        "type_json": f.json(),
        "type_name": map_type_name(f.dataType),
        "position": i,
        "nullable": f.nullable,
    }
    for i, f in enumerate(schema.fields)
]

resp = requests.post(
    "http://unity-catalog-server:8080/api/2.1/unity-catalog/tables",
    json={
        "name": "transformed_accounts",
        "catalog_name": "unity",
        "schema_name": "default",
        "table_type": "EXTERNAL",
        "data_source_format": "DELTA",
        "storage_location": uc_table_path,
        "columns": columns,
    },
)
print(f"Unity Catalog Registration Status: {resp.status_code}")
if resp.status_code not in (200, 201):
    print(f"Response: {resp.text}")
else:
    print(f"Successfully registered standard Delta table in Unity Catalog at {uc_table_path}.")



In [ ]:
# ------------------------------------------------------------------------------
# Phase 2E: CRUD Mutation Test (proves the reverse-sync in Phase 3 handles CHANGES,
# not just the initial load -- this was explicitly deferred as Phase 5/6 in plan.md
# and is a real, previously-untested gap until demonstrated here)


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2E] Applying CRUD changes to prove reverse-sync handles updates ---")

# UPDATE: give Bob a bonus (tests that a real UPDATE is correctly reflected downstream)
spark.sql(f"""
    UPDATE delta.`{uc_table_path}`
    SET balance = 9999, balance_with_interest = 9999 * 1.05
    WHERE account_id = '2'
""")
print("Applied UPDATE to account_id=2 (Bob).")

# DELETE: remove Charlie entirely (tests that a real DELETE is correctly reflected downstream)
spark.sql(f"""
    DELETE FROM delta.`{uc_table_path}`
    WHERE account_id = '3'
""")
print("Applied DELETE to account_id=3 (Charlie).")

print("\nCurrent state of the Unity Catalog Delta table after CRUD changes:")
spark.read.format("delta").load(uc_table_path).orderBy("account_id").show()


In [ ]:
# ------------------------------------------------------------------------------
print("\n--- [Phase 2E] Vacuuming Delta table before handing off to reverse-sync ---")
print(
    "NOTE: RETAIN 0 HOURS is used ONLY because this is a single-writer, serial\n"
    "POC with no concurrent readers or open time-travel queries. In production\n"
    "this is unsafe -- keep the default retention window (7 days) so concurrent\n"
    "reads and time travel aren't broken. This step exists purely so Phase 3's\n"
    "raw Parquet-directory scan doesn't pick up files superseded by the UPDATE/\n"
    "DELETE above (add_files has no awareness of Delta's transaction log; it\n"
    "just lists whatever physical files are sitting in the directory)."
)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM delta.`{uc_table_path}` RETAIN 0 HOURS")
print("Vacuum complete -- physical files now match the current logical snapshot.")

spark.stop()
print("\nPhase 2 Complete (including CRUD mutation test).")
